In [5]:
from langchain_community.document_loaders import (DirectoryLoader, PyPDFLoader, TextLoader, PythonLoader, Docx2txtLoader, UnstructuredWordDocumentLoader, UnstructuredMarkdownLoader, UnstructuredFileLoader, UnstructuredPDFLoader)
from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from operator import add as add_messages
from langchain_unstructured import UnstructuredLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_core.tools import tool

In [3]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")


In [ ]:
# Load all course materials (PDFs, txt, Code, etc.)

loader = DirectoryLoader(
    path="GT-Courses/",  # Root folder containing AI/, CN/, etc.
    glob=["**/*.pdf", "**/*.py", "**/*.txt", "**/*.docx", "**/*.md", "**/*.R", "**/*.Rmd"],
    show_progress=True,
    
    use_multithreading=True,
    silent_errors=True
)
file_loaders ={
    ".pdf": UnstructuredPDFLoader,
    ".py": PythonLoader,
    ".txt": TextLoader,
    ".docx": UnstructuredWordDocumentLoader,
    ".md": UnstructuredMarkdownLoader,
    ".R": TextLoader,
    ".Rmd": UnstructuredFileLoader
}

all_docs = []
for root, _, files in os.walk("GT-Courses/"):
    for file in files:
        ext = os.path.splitext(file)[1]
        if ext in file_loaders:
            try:
                print(ext)
                loader = file_loaders[ext](os.path.join(root, file))
                loaded = loader.load()
                # Add file path to metadata
                for doc in loaded:
                    doc.metadata["file_path"] = os.path.join(root, file)
                    doc.metadata["course"] = os.path.basename(root)
                all_docs.extend(loaded)
            except Exception as e:
                print(f"Error loading {file}: {e}")

In [ ]:
all_docs

In [ ]:
import time
from datetime import datetime
from tqdm import tqdm  # Progress bars
from tenacity import retry, stop_after_attempt, wait_exponential

# Initialize monitoring
class ProcessingMonitor:
    def __init__(self):
        self.start_time = time.time()
        self.processed_count = 0
        self.error_count = 0
        self.chunk_lengths = []
        
    def log_success(self, chunk_length):
        self.processed_count += 1
        self.chunk_lengths.append(chunk_length)
        
    def log_error(self, error_msg):
        self.error_count += 1
        with open("processing_errors.log", "a") as f:
            f.write(f"{datetime.now()}: {error_msg}\n")
    
    def get_stats(self):
        elapsed = time.time() - self.start_time
        avg_length = sum(self.chunk_lengths)/len(self.chunk_lengths) if self.chunk_lengths else 0
        return {
            "elapsed_min": round(elapsed/60, 2),
            "docs_processed": self.processed_count,
            "error_count": self.error_count,
            "avg_chunk_len": round(avg_length),
            "docs_remaining": len(splits) - self.processed_count - self.error_count
        }

# Configure text splitter with monitoring
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=800,
    separators=["\n\n", "\n", " "]
)

print("⚡ Starting document splitting...")
splits = text_splitter.split_documents(all_docs)
print(f"✅ Split {len(all_docs)} source docs into {len(splits)} chunks")

# Initialize embeddings with retry
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=60))
def get_embeddings():
    return GoogleGenerativeAIEmbeddings(model="models/embedding-001")

embeddings = get_embeddings()
monitor = ProcessingMonitor()

# Process with real-time monitoring
processed_chunks = []
with tqdm(total=len(splits), desc="📊 Embedding chunks") as pbar:
    for i, chunk in enumerate(splits):
        try:
            # Rate limiting (1 request/sec)
            # if i % 10 == 0:
            #     time.sleep(1)  
            
            # Track processing
            start_time = time.time()
            chunk_length = len(chunk.page_content)
            
            # Process chunk
            embedding = embeddings.embed_query(chunk.page_content)
            chunk.metadata.update({
                "embedding_time": round(time.time() - start_time, 2),
                "chunk_length": chunk_length,
                "processed_at": datetime.now().isoformat()
            })
            processed_chunks.append(chunk)
            
            # Update monitor
            monitor.log_success(chunk_length)
            pbar.update(1)
            pbar.set_postfix(monitor.get_stats())
            
        except Exception as e:
            monitor.log_error(f"Chunk {i} failed: {str(e)}")
            chunk.metadata["error"] = str(e)
            processed_chunks.append(chunk)
            pbar.update(1)



In [24]:
# Save to ChromaDB with validation
print("\n💾 Saving to ChromaDB...")
try:
    vector_db = Chroma.from_documents(
        documents=processed_chunks,
        embedding=embeddings,
        persist_directory="courses_db",
        collection_name="courses_768d"
    )
    
    # Verify
    print(f"🔍 Sample check: {vector_db._collection.count()} chunks stored")
    print("✅ Completed successfully!")
    print("\n📈 Final Stats:")
    stats = monitor.get_stats()
    for k, v in stats.items():
        print(f"- {k.replace('_', ' ').title()}: {v}")
        
except Exception as e:
    print(f"❌ Critical error saving to Chroma: {e}")
    with open("failed_chunks.json", "w") as f:
        import json
        json.dump([doc.metadata for doc in processed_chunks], f)
    print("⚠️ Saved metadata for recovery in failed_chunks.json")


💾 Saving to ChromaDB...
🔍 Sample check: 15515 chunks stored
✅ Completed successfully!

📈 Final Stats:
- Elapsed Min: 220.86
- Docs Processed: 15515
- Error Count: 0
- Avg Chunk Len: 4115
- Docs Remaining: 0


In [25]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [53]:
@tool
def retrieve_tool(query:str)->str:
    """
    This tool searches and returns the information from the course notes
    """
    docs = retriever.invoke(query)
    if not docs:
        return "I found no relevant information in the course notes"
    result = []
    for doc in docs:
        current = f"File path: {doc.metadata['file_path']}:\n{doc.page_content}"
    
        print(f"File path: {doc.metadata['file_path']}")
        # print(f"Document content: {doc.page_content}")
        result.append(current)
    return "\n\n".join(result)
        

In [54]:
tools = [retrieve_tool]
llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", google_api_key=api_key, temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [55]:
class AgentState(TypedDict):
     messages: Annotated[Sequence[BaseMessage], add_messages]

In [56]:
def should_continue(state: AgentState):
    """Check if the last message contains tool calls."""
    result = state['messages'][-1]
    return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0

In [57]:
system_prompt = """
You are an intelligent AI assistant who answers questions about GT course content based on the document loaded into your knowledge base.
Use the retriever tool available to answer questions about the course content. You can make multiple calls if needed.
If you need to look up some information before asking a follow up question, you are allowed to do that!
Please always cite the specific parts of the documents you use in your answers.
"""

In [58]:
tools_dict = {our_tool.name: our_tool for our_tool in tools} # Creating a dictionary of our tools

In [59]:
def call_llm(state: AgentState) -> AgentState:
    """Function to call the LLM with the current state."""
    messages = list(state['messages'])
    system_message = SystemMessage(content=system_prompt)
    if messages and isinstance(messages[0], SystemMessage):
        messages[0] = system_message
    else:
        
        messages = [system_message] + messages
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}

In [60]:
def take_action(state: AgentState) -> AgentState:
    """Execute tool calls from the LLM's response."""

    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")
        
        if not t['name'] in tools_dict: # Checks if a valid tool is present
            print(f"\nTool: {t['name']} does not exist.")
            result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."
        
        else:
            result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
            print(f"Result length: {len(str(result))}")
            

        # Appends the Tool Message
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

    print("Tools Execution Complete. Back to the model!")
    return {'messages': results}



In [61]:
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)

graph.add_conditional_edges(
    "llm",
    should_continue,
    {True: "retriever_agent", False: END}
)
graph.add_edge("retriever_agent", "llm")
graph.set_entry_point("llm")

rag_agent = graph.compile()

In [62]:
def running_agent():
    print("\n=== RAG AGENT===")
    
    while True:
        user_input = input("\nWhat is your question: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        messages = [HumanMessage(content=user_input)] # converts back to a HumanMessage type

        result = rag_agent.invoke({"messages": messages})
        
        print("\n=== ANSWER ===")
        if hasattr(result["messages"][-1], "content"):
            print(result['messages'][-1].content)
        else:
            print(result["messages"][-1])

# Running Questions

For example:
Q1: Find homeworks that are related to Optimization in IAM
Q2: Which model can explain the development of computer network

In [63]:
running_agent()


=== RAG AGENT===
Calling Tool: retrieve_tool with query: IAM homework related to Optimization
File path: GT-Courses/IAM/4homeworks/hw10/1hw_description/HW10 Header.pdf
File path: GT-Courses/IAM/0glossary/ISYE6501glossarybytopicFeb7_2018.pdf
File path: GT-Courses/IAM/4homeworks/hw11/1hw_description/HW11 Header.pdf
File path: GT-Courses/IAM/0glossary/ISYE6501glossarybytopicFeb7_2018.pdf
File path: GT-Courses/IAM/3transcripts/OMSA_ISyE6501_M15L6_ClassificationOptimizationMode-en.txt
Result length: 15256
Tools Execution Complete. Back to the model!

=== ANSWER ===
Based on the course materials, the following homework assignments are related to Optimization:

1.  **Homework 10 (Question 15.1):** This question asks you to describe a real-world situation or problem for which optimization would be an appropriate solution and to identify the data you would need.
    *   **Source:** `GT-Courses/IAM/4homeworks/hw10/1hw_description/HW10 Header.pdf`

2.  **Homework 11 (Question 15.2):** This assig